# ex02 · 广播与形状预测（对应教材 2.1 / 2.3）

> **广播规则复习**：从右往左对齐维度 → 缺维补 1 维 → 大小为 1 的维度自动拉伸 → 大小不同且都不为 1 则**报错**。
>
> **做题流程**：先写预测，再运行验证。这组题的目标是让你**只看代码就能说出输出形状**——这是以后 debug 报错 `shape mismatch` 的基本功。
> 答案在 `solutions/ex02-答案.md`。

In [2]:
import torch

## 题 1 🌱 `(2, 3) + (3,)`

`X = torch.ones(2, 3)`，`v = torch.tensor([10., 20., 30.])`

预测：结果形状 (2, 3)，内容（手算）torch.tensor([[11., 21., 31.],
                                               [11., 21., 31.]])

In [4]:
X = torch.ones(2, 3)
v = torch.tensor([10., 20., 30.])
print(X + v)

tensor([[11., 21., 31.],
        [11., 21., 31.]])


## 题 2 🌱 `(2, 1, 4) + (1, 3, 1)`

预测：结果形状 (2,3,4)。逐维分析：dim0: 2 vs 1 → 2；dim1: 1 vs 3 → 3；dim2: 4 vs 1 → 4

In [5]:
a = torch.ones(2, 1, 4)
b = torch.ones(1, 3, 1)
print((a + b).shape)

torch.Size([2, 3, 4])


## 题 3 🔧 预测「报错」题：`(2, 3, 4) + (2, 3)`

预测：能广播吗？不能 如果能，结果形状 ____；如果不能，哪一维冲突 1，2维其实都有问题，但是由于pytorch由右向左的检查所以只报出了dim2

In [6]:
a = torch.ones(2, 3, 4)
b = torch.ones(2, 3)
try:
    c = a + b
    print('成功了，形状 =', c.shape)
except Exception as e:
    print('报错：', e)

报错： The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 2


## 题 4 🌱 `(2, 3, 4) + (4,)`

一维向量广播到三维。预测：结果形状(2, 3, 4)，`(4,)` 被补成 (1, 1, 4)

In [7]:
a = torch.ones(2, 3, 4)
b = torch.arange(4.0)
print((a + b).shape)

torch.Size([2, 3, 4])


## 题 5 🔧 降维求和：`axis` 与 `keepdims`

`X = torch.arange(12).reshape(3, 4)`。预测下列结果的**形状**：

| 表达式 | 形状 |
|---|---|
| `X.sum(axis=0)` |(4,) |
| `X.sum(axis=1)` |(3,) |
| `X.sum(axis=0, keepdims=True)` |(1, 4) |
| `X.sum(axis=[0, 1])` |()相当于整体求和 |
| `X.mean(axis=1)` |(3,) |

In [8]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print(X.sum(axis=0).shape)
print(X.sum(axis=1).shape)
print(X.sum(axis=0, keepdims=True).shape)
print(X.sum(axis=[0, 1]).shape)
print(X.mean(axis=1).shape)

torch.Size([4])
torch.Size([3])
torch.Size([1, 4])
torch.Size([])
torch.Size([3])


## 题 6 🌱 手算广播：`(3, 5) + (5,)`

`X = torch.arange(15).reshape(3, 5)`，`v = torch.tensor([1., 2., 3., 4., 5.])`

预测 `X + v` 的**完整内容**（15 个数手算出来），再验证。
torch.tensor([[1., 3., 5., 7., 9.],
              [6., 8., 10., 12., 14.],
              [11., 13., 15., 17., 19.]])

In [9]:
X = torch.arange(15).reshape(3, 5)
v = torch.tensor([1., 2., 3., 4., 5.])
print(X + v)

tensor([[ 1.,  3.,  5.,  7.,  9.],
        [ 6.,  8., 10., 12., 14.],
        [11., 13., 15., 17., 19.]])


## 题 7 🔧 矩阵乘法的形状规则

矩阵乘法**不是**广播！`(2, 3) @ (3, 4)` 合法，`(2, 3) @ (4, 3)` 报错。预测下列哪些合法：

| 表达式 | 合法? | 结果形状 |
|---|---|---|
| `(2, 3) @ (3, 4)` |合法 |(2, 4) |
| `(3, 2) @ (2, 3)` |合法|(3, 3) |
| `(2, 3) @ (2, 3)` |不合法 | |
| `(3,) @ (3, 2)` |合法 |(2,) |
| `(2, 3, 4) @ (4, 5)` |合法 |(2, 3, 5) |

In [11]:
def try_matmul(a_shape, b_shape):
    a = torch.ones(*a_shape)
    b = torch.ones(*b_shape)
    try:
        c = a @ b
        print(f'{a_shape} @ {b_shape} → 合法，形状 {tuple(c.shape)}')
    except Exception as e:
        print(f'{a_shape} @ {b_shape} → 报错')

try_matmul((2, 3), (3, 4))
try_matmul((3, 2), (2, 3))
try_matmul((2, 3), (2, 3))
try_matmul((3,), (3, 2))
try_matmul((2, 3, 4), (4, 5))

(2, 3) @ (3, 4) → 合法，形状 (2, 4)
(3, 2) @ (2, 3) → 合法，形状 (3, 3)
(2, 3) @ (2, 3) → 报错
(3,) @ (3, 2) → 合法，形状 (2,)
(2, 3, 4) @ (4, 5) → 合法，形状 (2, 3, 5)


## 题 8 🔧 归一化技巧：`X / X.sum(1, keepdims=True)`

`X = torch.arange(12, dtype=torch.float32).reshape(3, 4)`。

1. 预测 `X.sum(1, keepdims=True)` 的形状和内容（先手算每行之和）
(3, 1)  torch.tensor([[6.],
                      [22.],
                      [38.]])   
3. 预测 `X / X.sum(1, keepdims=True)` 的内容（每行除以行和 → 行内和为 1）
每行分别除以对应行的行之和（每一行之和均为1）

**思考**：为什么必须加 `keepdims=True`，去掉会怎样？
会报错，因为（3，）会默认广播成（1，3）然后报错，因为第一维出现问题

In [12]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)
row_sum = X.sum(1, keepdims=True)
print('row_sum =\n', row_sum, '\n')
normalized = X / row_sum
print('normalized =\n', normalized, '\n')
print('每行之和（应全为 1）:\n', normalized.sum(1))

row_sum =
 tensor([[ 6.],
        [22.],
        [38.]]) 

normalized =
 tensor([[0.0000, 0.1667, 0.3333, 0.5000],
        [0.1818, 0.2273, 0.2727, 0.3182],
        [0.2105, 0.2368, 0.2632, 0.2895]]) 

每行之和（应全为 1）:
 tensor([1., 1., 1.])


## 题 9 🚀 挑战：两个维度同时广播

`a = torch.arange(3).reshape(1, 1, 3)`（形状 `(1,1,3)`），`b = torch.arange(4).reshape(1, 4, 1)`（形状 `(1,4,1)`）

手算 `a + b` 的完整结果（形状 `(1, 4, 3)`，12 个数），再验证。提示：结果等于 a 的每个值 + b 的每个值，排列成一个 4×3 的加法表。
torch.tensor([[0, 1, 2],
              [1, 2, 3],
              [2, 3, 4],
              [3, 4, 5]])

In [13]:
a = torch.arange(3).reshape(1, 1, 3)
b = torch.arange(4).reshape(1, 4, 1)
print('a =', a.flatten().tolist())
print('b =', b.flatten().tolist())
print('a + b =\n', a + b)

a = [0, 1, 2]
b = [0, 1, 2, 3]
a + b =
 tensor([[[0, 1, 2],
         [1, 2, 3],
         [2, 3, 4],
         [3, 4, 5]]])


## 题 10 🚀 挑战：解释这段代码为什么能跑

下面的代码计算「每张图每个通道的像素均值」（形状 `(2, 3)` 来自 `(2, 3, 4, 4)`）。

1. 写出每一行的输出形状
2. 用一句话解释 `mean(dim=[2, 3])` 做了什么

   是把第 2、3 轴（高和宽，共 16 个像素）同时消掉，剩下 `(2, 3)` = 每张图每个通道的像素均值。

In [15]:
imgs = torch.arange(2 * 3 * 4 * 4, dtype=torch.float32).reshape(2, 3, 4, 4)
print('imgs:        ', imgs.shape)                                     #torch.Size([2, 3, 4, 4])
print('mean(2,3):  ', imgs.mean(dim=[2, 3]).shape)                    #torch.Size([2, 3])
print('sum(0):     ', imgs.sum(dim=0).shape)                          #torch.Size([3, 4, 4])
print('sum(0,keepdims=True):', imgs.sum(dim=0, keepdims=True).shape)  #torch.Size([1, 3, 4, 4])

imgs:         torch.Size([2, 3, 4, 4])
mean(2,3):   torch.Size([2, 3])
sum(0):      torch.Size([3, 4, 4])
sum(0,keepdims=True): torch.Size([1, 3, 4, 4])
